# SoundVision AI — Exploratory Audio Analysis & Preprocessing Walkthrough

This notebook demonstrates the acoustic signal processing pipeline, dataset inspection, and Mel-spectrogram feature representations for the curated 10-category environmental sound benchmark from **ESC-50**.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from soundvision.config import load_config
from soundvision.data.dataset import SELECTED_CLASSES, CLASS_TO_IDX
from soundvision.data.preprocessing import AudioPreprocessor

print("Target Classes:", SELECTED_CLASSES)

## 1. Load Dataset Metadata & Inspect Fold Splits

In [ ]:
meta_path = Path("../data/raw/ESC-50-master/meta/esc50.csv")
if meta_path.is_file():
    df = pd.read_csv(meta_path)
    subset_df = df[df["category"].isin(SELECTED_CLASSES)].copy()
    print(f"Total subset recordings: {len(subset_df)}")
    print(subset_df.groupby(["category", "fold"]).size().unstack())
else:
    print("Dataset metadata not yet downloaded. Run `python scripts/download_dataset.py`.")

## 2. Audio Loading & Preprocessing Pipeline

In [ ]:
preprocessor = AudioPreprocessor(
    sample_rate=22050,
    n_fft=1024,
    hop_length=512,
    n_mels=128,
    target_duration=5.0,
)

# Create synthetic tone for demonstration
sr = 22050
t = np.linspace(0, 5.0, sr * 5)
synthetic_audio = 0.5 * np.sin(2 * np.pi * 440 * t) + 0.25 * np.sin(2 * np.pi * 880 * t)
wave_tensor = torch.from_numpy(synthetic_audio).float().unsqueeze(0)

mel_db = preprocessor.extract_mel_spectrogram(wave_tensor)
print("Mel-spectrogram shape:", mel_db.shape)
print("Min dB:", float(mel_db.min()), "Max dB:", float(mel_db.max()))

## 3. Waveform and Log-Mel Spectrogram Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))
ax1.plot(t[:1000], synthetic_audio[:1000])
ax1.set_title("Waveform (First 1000 samples)")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Amplitude")

im = ax2.imshow(mel_db.squeeze(0).numpy(), origin="lower", aspect="auto", cmap="magma")
ax2.set_title("Log-Mel Spectrogram (dB)")
ax2.set_xlabel("Time Frame")
ax2.set_ylabel("Mel Frequency Bin")
plt.colorbar(im, ax=ax2, label="dB")
plt.tight_layout()
plt.show()